# DESC ELAsTiCC2 — 06 : Entraînement du classifieur (Random Forest vs XGBoost)

Ce notebook relit `train_set.parquet` / `test_set.parquet` produits par
`05_build_train_test_sets.ipynb`, entraîne et compare un **Random Forest** et un
**XGBoost**, évalue les performances (accuracy, precision/recall/F1, matrice de
confusion, ROC-AUC), puis analyse l'importance des features (Random Forest +
valeurs SHAP).

Cible : classification binaire **SNIa vs non-Ia** (label produit par le notebook 05 ;
changer `TARGET_MODE='multiclass'` dans le notebook 05 et adapter les métriques
ci-dessous permettrait une classification multi-classe).

- author : Sylvie Dagoret-Campagne
- creation date : 2026-06-20
- last update : 2026-06-22
- input : `train_test_sets/train_set.parquet`, `train_test_sets/test_set.parquet` (notebook 05)

## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import pathlib
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, classification_report
)

_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)

# ── Dépendances optionnelles : XGBoost et SHAP ──────────────────────────────
# Le notebook reste pleinement utilisable (Random Forest + feature importance
# native) même si l'une ou l'autre n'est pas installée dans l'environnement.
try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    _logger.warning("xgboost non installé — comparaison XGBoost désactivée. "
                     "Installer avec : pip install xgboost")

try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    _logger.warning("shap non installé — analyse SHAP désactivée. "
                     "Installer avec : pip install shap")

_logger.info(f"Imports done. HAS_XGBOOST={HAS_XGBOOST}  HAS_SHAP={HAS_SHAP}")

In [ ]:
# to enlarge the sizes
params = {
    "legend.fontsize": "x-large",
    "figure.figsize": (10, 6),
    "axes.labelsize": "x-large",
    "axes.titlesize": "x-large",
    "xtick.labelsize": "x-large",
    "ytick.labelsize": "x-large",
}
plt.rcParams.update(params)

## 1 · Paramètres

- `TRAIN_TEST_DIR` : dossier contenant `train_set.parquet` / `test_set.parquet` (sortie du notebook 04).
- `RANDOM_SEED` : graine pour la reproductibilité des modèles.
- `N_ESTIMATORS_RF`, `MAX_DEPTH_RF` : hyperparamètres du Random Forest.
- `N_ESTIMATORS_XGB`, `MAX_DEPTH_XGB`, `LEARNING_RATE_XGB` : hyperparamètres de XGBoost.
- `OUTPUT_DIR` : dossier où sont sauvegardés les modèles entraînés et les figures.

In [ ]:
# ── Paramètres principaux ────────────────────────────────────────────────────

TRAIN_TEST_DIR = pathlib.Path(os.getcwd()) / "train_test_sets"

RANDOM_SEED = 42

N_ESTIMATORS_RF = 200
MAX_DEPTH_RF    = 10

N_ESTIMATORS_XGB   = 200
MAX_DEPTH_XGB       = 4
LEARNING_RATE_XGB   = 0.1

OUTPUT_DIR = pathlib.Path(os.getcwd()) / "models"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"TRAIN_TEST_DIR = {TRAIN_TEST_DIR}")
print(f"OUTPUT_DIR     = {OUTPUT_DIR}")

## 2 · Chargement des échantillons train/test

In [ ]:
train_path = TRAIN_TEST_DIR / "train_set.parquet"
test_path  = TRAIN_TEST_DIR / "test_set.parquet"

if not train_path.is_file() or not test_path.is_file():
    raise FileNotFoundError(
        f"train_set.parquet / test_set.parquet introuvables dans {TRAIN_TEST_DIR}. "
        f"Avez-vous exécuté 05_build_train_test_sets.ipynb ?"
    )

train_df = pd.read_parquet(train_path)
test_df  = pd.read_parquet(test_path)

print(f"Train : {train_df.shape}   Test : {test_df.shape}")
print("\nBalance des labels — Train :")
print(train_df['label_name'].value_counts(normalize=True).rename('fraction'))
print("\nBalance des labels — Test :")
print(test_df['label_name'].value_counts(normalize=True).rename('fraction'))

In [ ]:
# ── Colonnes de features : tout sauf les identifiants/labels ────────────────
non_feature_cols = {'SNID', 'obj_class', 'label', 'label_name', 'is_good_fit', 'z_bin'}
feature_cols = [c for c in train_df.columns if c not in non_feature_cols]

print(f"{len(feature_cols)} colonnes de features utilisées :")
print(feature_cols)

CLASS_NAMES = ['non-Ia', 'SNIa']  # label 0, label 1 (cf. notebook 05, TARGET_MODE='binary')

## 3 · Préparation X / y

Les NaN éventuels (bandes non ajustées pour un objet donné) sont remplacés par 0,
comme dans `extract_features` du module `classification` existant — un Random
Forest / Gradient Boosting peut en principe gérer du NaN nativement (XGBoost le
permet), mais on garde ce remplacement pour la cohérence avec le Random Forest
+ `StandardScaler`, qui lui ne supporte pas les NaN.

In [ ]:
X_train = train_df[feature_cols].fillna(0)
y_train = train_df['label'].astype(int)
X_test  = test_df[feature_cols].fillna(0)
y_test  = test_df['label'].astype(int)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"X_train : {X_train_scaled.shape}   X_test : {X_test_scaled.shape}")
n_nan_train = train_df[feature_cols].isna().sum().sum()
n_nan_test  = test_df[feature_cols].isna().sum().sum()
print(f"NaN remplacés par 0 — train : {n_nan_train}, test : {n_nan_test}")

## 4 · Entraînement : Random Forest

In [ ]:
rf_clf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS_RF,
    max_depth=MAX_DEPTH_RF,
    random_state=RANDOM_SEED,
    class_weight='balanced',
    n_jobs=-1,
)
rf_clf.fit(X_train_scaled, y_train)

y_pred_rf  = rf_clf.predict(X_test_scaled)
y_proba_rf = rf_clf.predict_proba(X_test_scaled)[:, 1]

print("Random Forest entraîné.")
print(classification_report(y_test, y_pred_rf, target_names=CLASS_NAMES))

## 5 · Entraînement : XGBoost (si disponible)

In [ ]:
if HAS_XGBOOST:
    xgb_clf = xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS_XGB,
        max_depth=MAX_DEPTH_XGB,
        learning_rate=LEARNING_RATE_XGB,
        random_state=RANDOM_SEED,
        eval_metric='logloss',
        n_jobs=-1,
    )
    xgb_clf.fit(X_train_scaled, y_train)

    y_pred_xgb  = xgb_clf.predict(X_test_scaled)
    y_proba_xgb = xgb_clf.predict_proba(X_test_scaled)[:, 1]

    print("XGBoost entraîné.")
    print(classification_report(y_test, y_pred_xgb, target_names=CLASS_NAMES))
else:
    xgb_clf = None
    y_pred_xgb, y_proba_xgb = None, None
    print("XGBoost non disponible — section ignorée.")

## 6 · Comparaison des métriques

Accuracy, precision, recall, F1 (classe positive = SNIa) et ROC-AUC, pour chaque modèle disponible.

### 6.0 Definition

#### 6.0.1) Precision (précision)

##### Définition

La precision mesure la fiabilité des prédictions positives :

$$ \text{Precision} = \frac{TP}{TP+FP}$$

- TP (True Positives) : vrais positifs (correctement détectés)
- FP (False Positives) : faux positifs (fausses alertes)
  
##### Interprétation

**Parmi ce que le modèle annonce comme positif, combien sont réellement positifs ?**

##### Intuition
Haute précision → peu de faux positifs

Utile quand les faux positifs coûtent cher (ex : alertes inutiles)

#### 6.0.2) Recall (rappel / sensibilité)

##### Définition

Le recall mesure la capacité à retrouver tous les vrais positifs :

$$\text{Recall} = \frac{TP}{TP+FN}$$

- FN (False Negatives) : faux négatifs (ratés)

##### Interprétation

**Parmi tous les vrais positifs, combien ont été détectés ?** (penser à l'efficacité de selection)

##### Intuition

- Haut recall → peu de faux négatifs
- Utile quand rater un signal est critique (ex : détection d’événements rares)

#### 6.0.3) Trade-off Precision / Recall

Ils sont souvent en tension via le seuil de décision :

Seuil bas → beaucoup de détections → recall ↑, precision ↓
Seuil haut → détections sûres → precision ↑, recall ↓

#### 6.0.4) Accuracy (exactitude globale)

##### Définition

$$ \text{Accuracy} = \frac{TP+TN}{TP+TN+FP+FN}$$

- TP : vrais positifs
- TN : vrais négatifs
- FP : faux positifs
- FN : faux négatifs

##### Interprétation

Quelle fraction des prédictions totales est correcte ?

C’est la métrique la plus intuitive :

elle regarde tout (positifs + négatifs)
elle donne une vision globale des performances

##### Intuition

Accuracy élevée → le modèle se trompe peu en moyenne
MAIS elle ne distingue pas le type d’erreur

##### Limite majeure : classes déséquilibrées

C’est le point crucial.

Exemple typique (astronomie / alertes)
- 99% = non-SN
- 1% = SN

Un modèle qui prédit toujours "non-SN" :

Accuracy=99%

→ mais :

- Recall = 0 (aucune SN détectée)
modèle inutilisable scientifiquement

#### 6.0.5) ROC curve (Receiver Operating Characteristic)

##### Définition

La courbe ROC trace :

$$ TPR=Recall \; \text{vs} \; FPR=\frac{FP}{FP+TN} $$


- TPR (True Positive Rate) = recall
- FPR (False Positive Rate) = taux de fausses alarmes
  
##### À quoi ça sert ?

On fait varier le seuil de classification, et on trace :

- $x = FPR$
- $y = TPR$

Chaque point = un seuil différent

##### Interprétation

Courbe proche du coin (0,1) → excellent modèle
Diagonale → modèle aléatoire
Plus la courbe est "bombée" vers le haut → meilleur

#####  AUC (Area Under Curve)

$$AUC=\int ROC $$

- 1.0 → parfait
- 0.5 → aléatoire
- < 0.5 → pire que random
  
##### Intuition probabiliste

Probabilité qu’un positif ait un score plus élevé qu’un négatif

| Métrique  | Sensibilité aux classes rares | Ce qu’elle mesure      |
| --------- | ----------------------------- | ---------------------- |
| Accuracy  | ❌ mauvaise                    | performance globale    |
| Precision | ✅                             | qualité des détections |
| Recall    | ✅                             | complétude             |
| ROC / AUC | ✅                             | compromis global       |


#### 6.0.6) Le F1-score est précisément conçu pour répondre au problème que tu rencontres implicitement : équilibrer precision et recall avec une seule métrique.

##### Définition

Le F1-score est la moyenne harmonique de la precision et du recall :

$$ F1=2 \times \frac{Precision \times Recall}{Precision+Recall} $$

	​

##### Pourquoi une moyenne harmonique ?

Contrairement à une moyenne classique, la moyenne harmonique :

- pénalise fortement les déséquilibres

Exemple :
Precision = 1.0
Recall = 0.0
F1=0

→ même si une métrique est parfaite, le score s’effondre si l’autre est mauvaise.

##### Interprétation

- Le F1-score est élevé uniquement si precision ET recall sont élevés.

F1 = 1 → parfait
F1 = 0 → catastrophique

##### Intuition

Le F1 répond à la question :

Mon modèle est-il à la fois fiable (precision) et complet (recall) ?

##### Cas typiques
✔️ Très utile si :
classes déséquilibrées
besoin d’un compromis entre :
faux positifs
faux négatifs

👉 typiquement : détection d’objets rares (SN, anomalies, transients)

##### Limites

Ne tient pas compte des vrais négatifs (TN)
Donc :
ignore une grande partie des données si classes très déséquilibrées
Ne remplace pas :
ROC / AUC
precision-recall curve

##### Variantes importantes

On peut généraliser :

$$ F_\beta = (1+\beta^2) \frac{PR}{P+R} $$

	​

F1 → équilibre (β = 1)

F2 → favorise recall

F0.5 → favorise precision

#####  Lecture rapide

Situation	Ce que fait F1
Precision haute, recall faible	↓ pénalisé
Recall haut, precision faible	↓ pénalisé
Les deux élevés	↑ bon score

In [ ]:
def compute_metrics(y_true, y_pred, y_proba, model_name):
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall': recall_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_proba),
    }


all_metrics = [compute_metrics(y_test, y_pred_rf, y_proba_rf, 'RandomForest')]
if HAS_XGBOOST:
    all_metrics.append(compute_metrics(y_test, y_pred_xgb, y_proba_xgb, 'XGBoost'))

comparison_df = pd.DataFrame(all_metrics).set_index('model').round(4)
comparison_df

### Matrices de confusion

In [ ]:
models_to_plot = [('Random Forest', y_pred_rf)]
if HAS_XGBOOST:
    models_to_plot.append(('XGBoost', y_pred_xgb))

fig, axes = plt.subplots(1, len(models_to_plot), figsize=(5.5 * len(models_to_plot), 4.5), tight_layout=True)
if len(models_to_plot) == 1:
    axes = [axes]

for ax, (name, y_pred) in zip(axes, models_to_plot):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title(name)

plt.show()

### Courbes ROC

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5.5), tight_layout=True)

fpr_rf, tpr_rf, _ = roc_curve(y_test, y_proba_rf)
auc_rf = roc_auc_score(y_test, y_proba_rf)
ax.plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={auc_rf:.3f})")

if HAS_XGBOOST:
    fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_proba_xgb)
    auc_xgb = roc_auc_score(y_test, y_proba_xgb)
    ax.plot(fpr_xgb, tpr_xgb, label=f"XGBoost (AUC={auc_xgb:.3f})")

ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Hasard')
ax.set_xlabel("Taux de faux positifs (FPR)")
ax.set_ylabel("Taux de vrais positifs (TPR)")
ax.set_title("Courbes ROC — SNIa vs non-Ia")
ax.legend(loc='lower right')
plt.show()

## 7 · Importance des features

### Random Forest (importance native, basée sur la diminution moyenne d'impureté)

In [ ]:
rf_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_clf.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

N_TOP = 20
fig, ax = plt.subplots(figsize=(7, 7), tight_layout=True)
top = rf_importance_df.head(N_TOP).iloc[::-1]
ax.barh(top['feature'], top['importance'])
ax.set_xlabel("Importance (Random Forest, diminution moyenne d'impureté)")
ax.set_title(f"Top {N_TOP} features — Random Forest")
plt.show()

rf_importance_df.head(N_TOP)

### SHAP (si disponible)

In [ ]:
if HAS_SHAP:
    explainer = shap.TreeExplainer(rf_clf)
    shap_values = explainer.shap_values(X_test_scaled)

    # Selon la version de shap, shap_values peut être :
    #  - une liste [shap_class0, shap_class1] (anciennes versions, classification binaire)
    #  - un array (n_samples, n_features, n_classes) (versions récentes)
    #  - un array (n_samples, n_features) (certains cas binaires)
    if isinstance(shap_values, list):
        sv_for_class1 = shap_values[1] if len(shap_values) > 1 else shap_values[0]
    elif shap_values.ndim == 3:
        sv_for_class1 = shap_values[:, :, 1]
    else:
        sv_for_class1 = shap_values

    mean_abs_shap = np.abs(sv_for_class1).mean(axis=0)
    shap_importance_df = pd.DataFrame({
        'feature': feature_cols,
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(7, 7), tight_layout=True)
    top_shap = shap_importance_df.head(N_TOP).iloc[::-1]
    ax.barh(top_shap['feature'], top_shap['mean_abs_shap'])
    ax.set_xlabel("Importance SHAP moyenne (|valeur|)")
    ax.set_title(f"Top {N_TOP} features — SHAP (classe SNIa)")
    plt.show()

    display(shap_importance_df.head(N_TOP))
else:
    shap_importance_df = None
    print("SHAP non disponible — section ignorée. Installer avec : pip install shap")

In [ ]:
if HAS_SHAP:
    # Beeswarm plot : distribution des valeurs SHAP par feature, colorée par valeur de la feature.
    # Donne une vue plus riche que le simple classement par importance moyenne.
    shap.summary_plot(
        sv_for_class1, X_test, feature_names=feature_cols,
        max_display=N_TOP, show=True
    )

## 8 · Sauvegarde des modèles et des résultats

Les modèles entraînés (`joblib`), le scaler, le tableau comparatif des métriques
et les tableaux d'importance des features sont sauvegardés pour réutilisation
(ex. application à de nouvelles données, rapport, notebook de suivi).

In [ ]:
import joblib

joblib.dump(rf_clf, OUTPUT_DIR / "random_forest_model.joblib")
joblib.dump(scaler, OUTPUT_DIR / "feature_scaler.joblib")
print(f"Random Forest + scaler sauvegardés dans {OUTPUT_DIR}")

if HAS_XGBOOST:
    joblib.dump(xgb_clf, OUTPUT_DIR / "xgboost_model.joblib")
    print(f"XGBoost sauvegardé dans {OUTPUT_DIR}")

comparison_df.to_csv(OUTPUT_DIR / "metrics_comparison.csv")
rf_importance_df.to_csv(OUTPUT_DIR / "rf_feature_importance.csv", index=False)
if HAS_SHAP:
    shap_importance_df.to_csv(OUTPUT_DIR / "shap_feature_importance.csv", index=False)

# Sauvegarde de la liste des features dans l'ordre exact utilisé à l'entraînement
# (indispensable pour appliquer le modèle à de nouvelles données plus tard).
with open(OUTPUT_DIR / "feature_columns.txt", "w") as f:
    f.write("\n".join(feature_cols))

print(f"\nMétriques et importances des features sauvegardées dans {OUTPUT_DIR}")

## 9 · Résumé

In [ ]:
print("Comparaison des modèles :")
display(comparison_df)

best_model_name = comparison_df['roc_auc'].idxmax()
print(f"\nMeilleur modèle (ROC-AUC) : {best_model_name}")

print("\nTop 5 features (Random Forest) :")
print(rf_importance_df.head(5).to_string(index=False))
if HAS_SHAP:
    print("\nTop 5 features (SHAP) :")
    print(shap_importance_df.head(5).to_string(index=False))